# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [46]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [47]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(message, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [48]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Hemingway.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [49]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To get to the other side.


In [50]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! It's nice to meet you. How are you doing today?


In [51]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have access to that information. However, feel free to share your name with me if you'd like!


In [52]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [53]:
import panel as pn  # GUI
def collect_messages(_): # signature - (_) is the event from pn.bind(button_conversation). 
    prompt = inp.value # captures the raw user-typed value; can also use inp.value
    inp.value = '' # resets the textbox after click
    if not prompt:
        # nothing enetere; just return what is there so far
        return pn.Column(*panels) if panels else pn.pane.Markdown("Say something to start!")
    context.append({'role':'user', 'content':f"{prompt}"}) # append user message to the chat context 
    # this grows the message history seeded earlier (with system prompt)
    response = get_completion_from_messages(context) # call model
    context.append({'role':'assistant', 'content':f"{response}"}) # appends assistant reply to chat history
    
    # render bubbles
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
    
    # return a layout to display
    return pn.Column(*panels)


In [54]:
import panel as pn  # GUI
pn.extension()  # initialize Panel assets (needed for notebooks/serve)

# --- conversation state ---
panels = []  # where we collect the rendered chat "bubbles"
context = [{
    'role': 'system',
    'content': """
You are OrderBot, an automated service to collect orders for a pizza restaurant.
You first greet the customer, then collect the order, and then ask if it's a pickup or delivery.
You wait to collect the entire order, then summarize it and check for a final
time if the customer wants to add anything else.
If it's a delivery, you ask for an address.
Finally you collect the payment.
Make sure to clarify all options, extras and sizes to uniquely identify the item from the menu.
You respond in a short, very conversational friendly style.
The menu includes
pepperoni pizza 12.95, 10.00, 7.00
cheese pizza 10.95, 9.25, 6.50
eggplant pizza 11.95, 9.75, 6.75
fries 4.50, 3.50
greek salad 7.25
Toppings:
extra cheese 2.00,
mushrooms 1.50
sausage 3.00
canadian bacon 3.50
AI sauce 1.50
peppers 1.00
Drinks:
coke 3.00, 2.00, 1.00
sprite 3.00, 2.00, 1.00
bottled water 5.00
"""
}]

# --- widgets ---
inp = pn.widgets.TextInput(value="Hi", placeholder="Enter text here…")  # user types here
button_conversation = pn.widgets.Button(name="Chat!")  # click to send

# --- model call (stub) ---
# Replace this with your real OpenAI call if desired.
def get_completion_from_messages(messages):
    # Very simple behavior: greet if no prior user message; otherwise echo + ask a clarifying question.
    user_msgs = [m["content"] for m in messages if m["role"] == "user"]
    if not user_msgs:
        return "Hi there! What would you like to order today?"
    last = user_msgs[-1]
    return (
        f"Got it: '{last}'. Could you tell me sizes (small/medium/large) and "
        f"whether this is pickup or delivery?"
    )

# --- callback ---
def collect_messages(_=None):
    prompt = (inp.value or "").strip()  # read from input
    inp.value = ""  # clear the box

    # if nothing entered; just return what we have so far
    if not prompt:
        return pn.Column(*(panels or [pn.pane.Markdown("Say something to start!")]),
                         sizing_mode="stretch_width")

    # append user to context
    context.append({'role': 'user', 'content': prompt})

    # call "model"
    try:
        response = get_completion_from_messages(context)
    except Exception as e:
        response = f"Oops — model call failed: {e}"

    # append assistant to context
    context.append({'role': 'assistant', 'content': response})

    # render chat bubbles
    panels.append(pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row(
            'Assistant:',
            pn.pane.Markdown(
                response, width=600,
                styles={'background-color': '#F6F6F6', 'padding': '6px', 'border-radius': '8px'}
            )
        )
    )

    # return a Panel layout (never wrap Panel layouts in Markdown)
    return pn.Column(*panels, sizing_mode="stretch_width")

# --- bind + layout ---
interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

# In a notebook, leave this as the last line of the cell to render the UI.
# From a terminal, you can run: panel serve your_script.py --autoreload
dashboard

from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
client = OpenAI()

def get_completion_from_messages(messages, model="gpt-4o-mini", temperature=0.4):
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content

dashboard


BokehModel(combine_events=True, render_bundle={'docs_json': {'6005c8cb-602f-4449-a4ed-2807208ba702': {'version…

In [ ]:
import panel as pn  # GUI
pn.extension()  # initialize Panel assets (needed for notebooks/serve)

# --- conversation state ---
panels = []  # where we collect the rendered chat "bubbles"
context = [{
    'role': 'system',
    'content': """
You are OrderBot, a witty but kind pizza-ordering assistant.
Use light sarcasm and playful humor, but NEVER be mean or confusing.
Still follow the exact flow: greet, collect full order with clarifications,
ask pickup/delivery (and address if delivery), summarize, confirm, then payment.
Do not skip confirmations due to jokes.
The menu includes
pepperoni pizza 12.95, 10.00, 7.00
cheese pizza 10.95, 9.25, 6.50
eggplant pizza 11.95, 9.75, 6.75
fries 4.50, 3.50
greek salad 7.25
Toppings:
extra cheese 2.00,
mushrooms 1.50
sausage 3.00
canadian bacon 3.50
AI sauce 1.50
peppers 1.00
Drinks:
coke 3.00, 2.00, 1.00
sprite 3.00, 2.00, 1.00
bottled water 5.00
"""
}]

# --- widgets ---
inp = pn.widgets.TextInput(value="Hi", placeholder="Enter text here…")  # user types here
button_conversation = pn.widgets.Button(name="Chat!")  # click to send

# --- model call (stub) ---
# Replace this with your real OpenAI call if desired.
def get_completion_from_messages(messages):
    # Very simple behavior: greet if no prior user message; otherwise echo + ask a clarifying question.
    user_msgs = [m["content"] for m in messages if m["role"] == "user"]
    if not user_msgs:
        return "Hi there! What would you like to order today?"
    last = user_msgs[-1]
    return (
        f"Got it: '{last}'. Could you tell me sizes (small/medium/large) and "
        f"whether this is pickup or delivery?"
    )

# --- callback ---
def collect_messages(_=None):
    prompt = (inp.value or "").strip()  # read from input
    inp.value = ""  # clear the box

    # if nothing entered; just return what we have so far
    if not prompt:
        return pn.Column(*(panels or [pn.pane.Markdown("Say something to start!")]),
                         sizing_mode="stretch_width")

    # append user to context
    context.append({'role': 'user', 'content': prompt})

    # call "model"
    try:
        response = get_completion_from_messages(context)
    except Exception as e:
        response = f"Oops — model call failed: {e}"

    # append assistant to context
    context.append({'role': 'assistant', 'content': response})

    # render chat bubbles
    panels.append(pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row(
            'Assistant:',
            pn.pane.Markdown(
                response, width=600,
                styles={'background-color': '#F6F6F6', 'padding': '6px', 'border-radius': '8px'}
            )
        )
    )

    # return a Panel layout (never wrap Panel layouts in Markdown)
    return pn.Column(*panels, sizing_mode="stretch_width")

# --- bind + layout ---
interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

# In a notebook, leave this as the last line of the cell to render the UI.
# From a terminal, you can run: panel serve your_script.py --autoreload
dashboard

from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
client = OpenAI()

def get_completion_from_messages(messages, model="gpt-4o-mini", temperature=0.4):
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content

dashboard


BokehModel(combine_events=True, render_bundle={'docs_json': {'41d83e54-c609-4936-b792-87da30b26a2c': {'version…

In [69]:
import panel as pn  # GUI
pn.extension()  # initialize Panel assets (needed for notebooks/serve)

# --- conversation state ---
panels = []  # where we collect the rendered chat "bubbles"
context = [{
    'role': 'system',
    'content': """
Goal: ask the MINIMUM number of questions while still ensuring a correct order.
Only ask for missing critical details (size, toppings, pickup vs delivery, address if delivery).
If the user implies defaults, confirm quickly.
Always summarize once and ask for a final add-ons sweep, then payment.
Be concise and efficient.
The menu includes
pepperoni pizza 12.95, 10.00, 7.00
cheese pizza 10.95, 9.25, 6.50
eggplant pizza 11.95, 9.75, 6.75
fries 4.50, 3.50
greek salad 7.25
Toppings:
extra cheese 2.00,
mushrooms 1.50
sausage 3.00
canadian bacon 3.50
AI sauce 1.50
peppers 1.00
Drinks:
coke 3.00, 2.00, 1.00
sprite 3.00, 2.00, 1.00
bottled water 5.00
"""
}]

# --- widgets ---
inp = pn.widgets.TextInput(value="Hi", placeholder="Enter text here…")  # user types here
button_conversation = pn.widgets.Button(name="Chat!")  # click to send

# --- model call (stub) ---
# Replace this with your real OpenAI call if desired.
def get_completion_from_messages(messages):
    # Very simple behavior: greet if no prior user message; otherwise echo + ask a clarifying question.
    user_msgs = [m["content"] for m in messages if m["role"] == "user"]
    if not user_msgs:
        return "Hi there! What would you like to order today?"
    last = user_msgs[-1]
    return (
        f"Got it: '{last}'. Could you tell me sizes (small/medium/large) and "
        f"whether this is pickup or delivery?"
    )

# --- callback ---
def collect_messages(_=None):
    prompt = (inp.value or "").strip()  # read from input
    inp.value = ""  # clear the box

    # if nothing entered; just return what we have so far
    if not prompt:
        return pn.Column(*(panels or [pn.pane.Markdown("Say something to start!")]),
                         sizing_mode="stretch_width")

    # append user to context
    context.append({'role': 'user', 'content': prompt})

    # call "model"
    try:
        response = get_completion_from_messages(context)
    except Exception as e:
        response = f"Oops — model call failed: {e}"

    # append assistant to context
    context.append({'role': 'assistant', 'content': response})

    # render chat bubbles
    panels.append(pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row(
            'Assistant:',
            pn.pane.Markdown(
                response, width=600,
                styles={'background-color': '#F6F6F6', 'padding': '6px', 'border-radius': '8px'}
            )
        )
    )

    # return a Panel layout (never wrap Panel layouts in Markdown)
    return pn.Column(*panels, sizing_mode="stretch_width")

# --- bind + layout ---
interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

# In a notebook, leave this as the last line of the cell to render the UI.
# From a terminal, you can run: panel serve your_script.py --autoreload
dashboard

from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
client = OpenAI()

def get_completion_from_messages(messages, model="gpt-4o-mini", temperature=0.4):
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content

dashboard


BokehModel(combine_events=True, render_bundle={'docs_json': {'d353084d-bf17-4a19-85d6-59f59b400a1a': {'version…

In [71]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

Here's the summary of your order in JSON format:

```json
{
  "pizza": {
    "type": "pepperoni",
    "size": "small",
    "price": 10.00
  },
  "toppings": [],
  "drinks": [],
  "sides": [],
  "total_price": 10.00
}
```

Would you like to proceed with this order? If so, please provide the delivery address for the pizza.


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [56]:
import panel as pn
pn.extension()

col = pn.Column("# Hello", pn.widgets.Button(name="Click me"))
col  # <-- must render here; do NOT wrap in Markdown()


BokehModel(combine_events=True, render_bundle={'docs_json': {'7838ddc4-0106-4201-9b05-2c01b7272f73': {'version…

# orderbot_variants

In [57]:
import panel as pn
pn.extension()

In [58]:
import os
import panel as pn
from dotenv import load_dotenv
from openai import OpenAI

# ---------- Setup ----------
load_dotenv()                 # reads .env (expects OPENAI_API_KEY)
client = OpenAI()             # uses OPENAI_API_KEY from env
pn.extension()                # loads Panel assets (in notebooks, ensure jupyter_bokeh is installed)

MENU_TEXT = """
pepperoni pizza 12.95, 10.00, 7.00
cheese pizza 10.95, 9.25, 6.50
eggplant pizza 11.95, 9.75, 6.75
fries 4.50, 3.50
greek salad 7.25
Toppings: extra cheese 2.00, mushrooms 1.50, sausage 3.00, canadian bacon 3.50, AI sauce 1.50, peppers 1.00
Drinks: coke 3.00, 2.00, 1.00; sprite 3.00, 2.00, 1.00; bottled water 5.00
""".strip()

# ---------- System prompts (3 personalities) ----------
SYSTEM_STANDARD = f"""You are OrderBot, an automated service to collect orders for a pizza restaurant.
Greet the customer, collect the entire order, then ask if it's pickup or delivery.
If delivery, ask for the address. Summarize the order and confirm once before payment.
Always clarify sizes, toppings, and drink sizes to uniquely identify items.
Be brief, friendly, and conversational.
Menu:
{MENU_TEXT}
"""

SYSTEM_HUMOR = f"""You are OrderBot, a witty but kind pizza-ordering assistant.
Use light sarcasm and playful humor, but NEVER be mean or confusing.
Still follow the exact flow: greet, collect full order with clarifications,
ask pickup/delivery (and address if delivery), summarize, confirm, then payment.
Do not skip confirmations due to jokes.
Menu:
{MENU_TEXT}
"""

SYSTEM_SPEED = f"""You are OrderBot in SPEED MODE.
Goal: ask the MINIMUM number of questions while still ensuring a correct order.
Only ask for missing critical details (size, toppings, pickup vs delivery, address if delivery).
If the user implies defaults, confirm quickly.
Always summarize once and ask for a final add-ons sweep, then payment.
Be concise and efficient.
Menu:
{MENU_TEXT}
"""

# ---------- Model call helper ----------
def chat_completion(messages, model="gpt-4o-mini", temperature=0.4):
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content

# ---------- Builder for one bot ----------
def build_orderbot(system_prompt: str, title: str):
    context = [{"role": "system", "content": system_prompt}]
    bubbles = []
    MAX_TURNS = 12  # keep history bounded (system + last 12 turns)

    inp = pn.widgets.TextInput(placeholder="Say hi or start your order…")
    btn = pn.widgets.Button(name="Chat", button_type="primary")
    clear_btn = pn.widgets.Button(name="Clear")

    def render_bubbles():
        if not bubbles:
            return pn.Column(
            pn.pane.Markdown("_No messages yet — say hello!_"),
            sizing_mode="stretch_width"
            )
        return pn.Column(*bubbles, sizing_mode="stretch_width")


    def reset():
        bubbles.clear()
        del context[1:]  # keep system prompt
        inp.value = ""

    def on_clear(_=None):
        reset()
        chat_panel.object = render_bubbles()

    def send_message(_=None):
        user_text = (inp.value or "").strip()
        inp.value = ""
        if not user_text:
            return

        # show user bubble
        bubbles.append(pn.Row("**You:**", pn.pane.Markdown(user_text, width=650)))

        # prepare trimmed context
        trimmed = [context[0]] + context[-2*MAX_TURNS:]
        trimmed.append({"role": "user", "content": user_text})

        # call model
        try:
            reply = chat_completion(trimmed)
        except Exception as e:
            reply = f"Sorry, I hit an error talking to the model: {e}"

        # update full context
        context.append({"role": "user", "content": user_text})
        context.append({"role": "assistant", "content": reply})

        # show assistant bubble
        bubbles.append(
            pn.Row(
                "**OrderBot:**",
                pn.pane.Markdown(
                    reply,
                    width=650,
                    styles={"background-color": "#F6F6F6", "padding": "8px", "border-radius": "10px"},
                ),
            )
        )
        chat_panel.object = render_bubbles()

    # wire events
    btn.on_click(send_message)
    clear_btn.on_click(on_clear)
    # Pressing Enter in the input triggers send
    inp.param.watch(lambda *_: None, "value")  # noop to keep reactive
    inp.jscallback(args=dict(btn=btn), value="""
      // When Enter pressed in the input, click the button
      if (cb_obj.value !== undefined) { /* noop */ }
    """)
    # A simple workaround: use KeyDown to submit on Enter via pn.widgets.Button
    # (if above doesn't work in your env, just click the button)

    chat_panel = pn.panel(render_bubbles(), loading_indicator=True, height=360)

    header = pn.Row(pn.pane.Markdown(f"### {title}"), pn.Spacer(), clear_btn)

    return pn.Column(
        header,
        pn.Row(inp, btn, sizing_mode="stretch_width"),
        chat_panel,
        sizing_mode="stretch_width",
        margin=(10, 10),
    )

# ---------- Build three variants ----------
standard_bot = build_orderbot(SYSTEM_STANDARD, "Standard — Friendly & Structured")
humor_bot    = build_orderbot(SYSTEM_HUMOR,   "Humorous/Sarcastic — Playful but Precise")
speed_bot    = build_orderbot(SYSTEM_SPEED,   "Speed-Mode — Minimal Questions")

# ---------- Tabs app ----------
app = pn.Tabs(
    ("Standard", standard_bot),
    ("Humorous", humor_bot),
    ("Speed-Mode", speed_bot),
    tabs_location="above",
)

# For notebooks: display `app`; for server: serve this file.
app.servable()

app


BokehModel(combine_events=True, render_bundle={'docs_json': {'2bfb3679-cb90-4a69-8cbf-a8b5a8be3f18': {'version…

In [60]:
from openai import OpenAI
client = OpenAI()

def chat_completion(messages, model="gpt-4o-mini", temperature=0.4):
    resp = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature
    )
    return resp.choices[0].message.content


Part 1 — Three Prompt Variations
Prompt Version 1 — Straightforward Restaurant Order Bot

"You are a pizza ordering assistant. Greet the customer, take their order, ask clarifying questions, confirm, and finalize the order.

Result: Worked very well. Bot asked about size, toppings, and pickup/delivery. Clear and polite.

Prompt Version 2 — Make the Bot Funny + Sarcastic

Be a sarcastic but friendly pizza-ordering robot. Light humor allowed, but still accurately take orders and confirm details. Do not insult customers.

Example Behavior:
Customer: I want a pepperoni pizza
Bot: Oh wow… pepperoni? How original. One pepperoni pizza coming up — what size this groundbreaking pizza should be?

Result:
Entertaining, but sometimes sarcasm interfered with clarity (e.g., it forgot to confirm drink orders once). Could mildly confuse users.

Prompt Version 3 — “Speed-Mode Order Bot” (as few questions as possible)

You are a fast-action pizza ordering system. Ask the minimum number of questions needed. Summarize clearly, then check once for additions.

Result:
Good for quick ordering; however, once it skipped confirming toppings and defaulted to “no extra toppings” incorrectly.

Part 2 — One-Page Summary
Report: Prompt Engineering for OrderBot

The goal of this exercise was to explore prompt engineering by modifying the behavior of a pizza ordering chatbot. I created three prompt variations: a standard friendly assistant, a humorous/sarcastic version, and a fast-mode assistant designed to minimize questions.

The standard prompt performed the most reliably. It clarified the order, asked about sizes, toppings, and delivery options, and always summarized the order before confirming. This aligned closely with how a real ordering system should behave. Accuracy was consistent and no hallucinations occurred.

The humorous/sarcastic prompt introduced personality, and responses were engaging. However, humor sometimes interfered with task accuracy. In a few interactions, the bot forgot to confirm drink size or skipped the delivery question. This demonstrates that adding style or personality can reduce precision unless constraints are clear.

The speed-mode prompt made ordering faster and asked fewer questions, but it assumed defaults when the user did not mention details. This resulted in errors, such as incorrectly assuming “no toppings.” It was efficient but not always reliable. The takeaway is that reducing clarifying steps improves speed but increases chances of mistakes.

What I Learned

This exercise showed how prompt design directly affects chatbot behavior, clarity, and accuracy. Adding personality increases engagement but risks losing structure. Reducing questions boosts speed but hurts reliability. Clear system instructions produce the most stable results.

This taught me that effective prompt engineering requires balancing personality, clarity, and constraints. Testing variations is essential to detect hallucinations or gaps. A well-designed prompt acts like a set of rules — the AI will follow them closely, so wording matters. Ultimately, prompt precision leads to consistent and dependable performance.